In [2]:
import pandas as pd

# ---- EDIT THIS ----
INPUT_PATH = "C:\\Users\\USER\\Desktop\\MediRAG\\data division logic\\miriad_neurology.jsonl"  
OUTPUT_CSV = "neurology_10k.csv"
OUTPUT_JSONL = "neurology_10k.jsonl"
# -------------------

# Load JSONL
df = pd.read_json(INPUT_PATH, lines=True)
print("Loaded rows:", len(df))

# Required columns
required_cols = ["question", "answer", "specialty"]
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

print("\nSpecialty distribution:")
print(df["specialty"].value_counts().head(10))

# Filter for the two specializations
df_filtered = df[df["specialty"].isin(["Neurology", "Neurosurgery"])]
print("\nFiltered rows (Neurology + Neurosurgery):", len(df_filtered))

# Sample 5k each
sample_size = 5000

neurology_df = df_filtered[df_filtered["specialty"] == "Neurology"].sample(
    n=sample_size, random_state=42
)

neurosurg_df = df_filtered[df_filtered["specialty"] == "Neurosurgery"].sample(
    n=sample_size, random_state=42
)

# Combine
balanced_df = pd.concat([neurology_df, neurosurg_df], ignore_index=True)
print("\nFinal balanced dataset size:", len(balanced_df))

# Shuffle
balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Save CSV
balanced_df.to_csv(OUTPUT_CSV, index=False)
print("Saved CSV:", OUTPUT_CSV)

# Save JSONL
balanced_df.to_json(OUTPUT_JSONL, orient="records", lines=True)
print("Saved JSONL:", OUTPUT_JSONL)

# Quick preview
balanced_df.head()


Loaded rows: 205154

Specialty distribution:
specialty
Neurology       198125
Neurosurgery      7029
Name: count, dtype: int64

Filtered rows (Neurology + Neurosurgery): 205154

Final balanced dataset size: 10000
Saved CSV: neurology_10k.csv
Saved JSONL: neurology_10k.jsonl


,qa_id,paper_id,question,answer,paper_url,paper_title,passage_text,passage_position,year,venue,specialty
0,4520349766631,203497666,What are the advantages of minimally invasive ...,Minimally invasive surgical techniques for int...,https://api.semanticscholar.org/CorpusID:20349...,Endoscopic surgery for thalamic hemorrhage bre...,"In this way, not only the hematoma can be clea...",3,2019.0,Chinese Journal of Traumatology,Neurosurgery
1,301795582803,17955828,How do BK channels play a role in the excitabi...,BK channels have been implicated in the excita...,https://api.semanticscholar.org/CorpusID:17955828,Functional Upregulation of Ca2+ -Activated K+ ...,The physiology of dopamine neurons of the subs...,0,2012.0,PLoS ONE,Neurology
2,6520793892801,207938928,What are the different grades of glioma and ho...,Glioma is subdivided into WHO I to IV grades. ...,https://api.semanticscholar.org/CorpusID:20793...,Glioma SOX2 expression decreased after adjuvan...,Glioma is the most common primary intracranial...,0,2019.0,BMC Cancer,Neurology
3,4972283551,9722835,How does diffusion imaging help in understandi...,"Diffusion imaging, including diffusion-weighte...",https://api.semanticscholar.org/CorpusID:9722835,MRI in multiple sclerosis: What’s inside the t...,"93, 96, 97 These changes may appear very early...",5,2011.0,Neurotherapeutics,Neurology
4,8821041321,210413,What is the minimal clinically important chang...,The minimal CID for the UPDRS motor score is a...,https://api.semanticscholar.org/CorpusID:210413,The Clinically Important Difference on the Uni...,Regression analysis showed the average differe...,2,2010.0,Archives of neurology,Neurology


In [ ]:
import pandas as pd

INPUT_CSV = "neurology_10k.csv"
INPUT_JSONL = "neurology_10k.jsonl"

OUTPUT_CSV_MIN = "neurology_cleaned.csv"
OUTPUT_JSONL_MIN = "neurology_cleaned.jsonl"

# Load CSV or JSONL (JSONL preferred because it's cleaner)
df = pd.read_json(INPUT_JSONL, lines=True)

print("Columns before:", df.columns.tolist())

# Keep only necessary columns
cols_to_keep = ["question", "answer", "specialty"]  # remove 'specialty' here if you truly don't want it

df_min = df[cols_to_keep].copy()

print("Columns after:", df_min.columns.tolist())
print("Final dataset size:", len(df_min))

# Save minimal CSV
df_min.to_csv(OUTPUT_CSV_MIN, index=False)
print("Saved minimal CSV:", OUTPUT_CSV_MIN)

# Save minimal JSONL
df_min.to_json(OUTPUT_JSONL_MIN, orient="records", lines=True)
print("Saved minimal JSONL:", OUTPUT_JSONL_MIN)

# Preview
df_min.head()


Columns before: ['qa_id', 'paper_id', 'question', 'answer', 'paper_url', 'paper_title', 'passage_text', 'passage_position', 'year', 'venue', 'specialty']
Columns after: ['question', 'answer', 'specialty']
Final dataset size: 10000
Saved minimal CSV: neurology_cleaned.csv
Saved minimal JSONL: neurology_cleaned.jsonl


,question,answer,specialty
0,What are the advantages of minimally invasive ...,Minimally invasive surgical techniques for int...,Neurosurgery
1,How do BK channels play a role in the excitabi...,BK channels have been implicated in the excita...,Neurology
2,What are the different grades of glioma and ho...,Glioma is subdivided into WHO I to IV grades. ...,Neurology
3,How does diffusion imaging help in understandi...,"Diffusion imaging, including diffusion-weighte...",Neurology
4,What is the minimal clinically important chang...,The minimal CID for the UPDRS motor score is a...,Neurology


In [2]:
from openai import OpenAI
import pandas as pd
from tqdm import tqdm
import time
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

if api_key is None:
    raise ValueError("OPENAI_API_KEY not found in .env file")

# Initialize GPT-4o client
client = OpenAI(api_key=api_key)

print("OpenAI client initialized successfully.")


OpenAI client initialized successfully.


In [3]:
def build_rewriter_prompt(question):
    prompt = f"""
You are a medical information retrieval assistant specializing in Neurology and Neurosurgery.
Your task is to rewrite a medical question into short, search-friendly queries that work well for
retrieving scientific and clinical information.

Guidelines:
- Keep queries short and keyword-focused.
- Preserve essential neurological and neurosurgical terminology.
- Remove filler words.
- Provide 2 variations separated by semicolons.
- Do NOT answer the question. Only rewrite it.
- End the output with "**".

Example:
Question: What are the advantages of minimally invasive surgical techniques for the treatment of intracerebral hematomas compared to craniotomy with hematoma evacuation?
Rewritten Queries: advantages minimally invasive surgery intracerebral hematoma; MIS vs craniotomy intracerebral hematoma outcomes**

Now rewrite the following question:
Question: {question}
Rewritten Queries:
"""
    return prompt.strip()



In [10]:
def generate_rewrites(question):
    prompt = build_rewriter_prompt(question)

    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.2
        )

        text = response.choices[0].message.content

        # Clean + split rewrites
        rewrites = [r.strip().replace("**","") for r in text.split(";") if r.strip()]
        return rewrites

    except Exception as e:
        print("Error:", e)
        return []


In [11]:
df = pd.read_json("neurology_cleaned.jsonl", lines=True)
print("Rows loaded:", len(df))
df.head()


Rows loaded: 10000


,question,answer,specialty
0,What are the advantages of minimally invasive ...,Minimally invasive surgical techniques for int...,Neurosurgery
1,How do BK channels play a role in the excitabi...,BK channels have been implicated in the excita...,Neurology
2,What are the different grades of glioma and ho...,Glioma is subdivided into WHO I to IV grades. ...,Neurology
3,How does diffusion imaging help in understandi...,"Diffusion imaging, including diffusion-weighte...",Neurology
4,What is the minimal clinically important chang...,The minimal CID for the UPDRS motor score is a...,Neurology


In [13]:
OUTPUT_PARTIAL = "neurology_partial_rewrites.jsonl"

# If partial file exists, load it and continue
if os.path.exists(OUTPUT_PARTIAL):
    df_partial = pd.read_json(OUTPUT_PARTIAL, lines=True)
    df = df_partial
    print("Resuming from partial file!")
else:
    df["rewrites"] = None
    print("Starting fresh rewriting job.")


Starting fresh rewriting job.


In [14]:
checkpoint_every = 200   # save progress every 200 questions

for idx in tqdm(range(len(df)), desc="Rewriting queries"):
    # Skip rows that already have rewrites (resume mode)
    if isinstance(df.loc[idx, "rewrites"], list):
        continue

    q = df.loc[idx, "question"]
    rewrites = generate_rewrites(q)

    # Retry once if empty
    if not rewrites:
        time.sleep(1)
        rewrites = generate_rewrites(q)

    df.at[idx, "rewrites"] = rewrites

    # Checkpoint save
    if idx % checkpoint_every == 0:
        df.to_json(OUTPUT_PARTIAL, orient="records", lines=True)
        print(f"Checkpoint saved at row {idx}")

# Save final file
df.to_json("neurology_10k_with_rewrites.jsonl", orient="records", lines=True)
print("🎉 Done! Final dataset saved.") 


Rewriting queries:   0%|          | 1/10000 [00:00<2:31:20,  1.10it/s]

Checkpoint saved at row 0


Rewriting queries:   2%|▏         | 201/10000 [03:04<2:38:22,  1.03it/s]

Checkpoint saved at row 200


Rewriting queries:   4%|▍         | 401/10000 [06:05<2:09:48,  1.23it/s]

...

Checkpoint saved at row 9600


Rewriting queries:  98%|█████████▊| 9801/10000 [2:29:51<03:13,  1.03it/s]

Checkpoint saved at row 9800


Rewriting queries: 100%|██████████| 10000/10000 [2:32:51<00:00,  1.09it/s]

🎉 Done! Final dataset saved.


In [1]:
# Substep 3.1: Build FAISS index of all "answer" texts
# Run this cell in your notebook. It saves index and metadata to disk.

# Install packages if not present (uncomment if needed)
# !pip install -q sentence-transformers faiss-cpu tqdm

from sentence_transformers import SentenceTransformer
import faiss
import pandas as pd
import numpy as np
import json
from tqdm import tqdm
import os

# ---- EDIT paths if needed ----
INPUT_JSONL = "neurology_10k_with_rewrites.jsonl"   # the file you saved after Step 2
FAISS_INDEX_PATH = "answers_faiss.index"
META_PATH = "answers_metadata.json"
EMBEDDING_MODEL_NAME = "all-mpnet-base-v2"  # good general purpose SBERT; change if you prefer
BATCH_SIZE = 64
# ------------------------------

# 1. Load dataset
print("Loading dataset:", INPUT_JSONL)
df = pd.read_json(INPUT_JSONL, lines=True)
N = len(df)
print("Rows loaded:", N)

# 2. Extract answers and basic metadata
answers = df["answer"].astype(str).tolist()   # ensure strings
# We'll use row index as doc_id
doc_ids = list(range(N))

# Optional: keep mapping metadata to recover original row and check matches
# We'll store: doc_id -> {index_in_df, question_preview, answer_preview, specialty}
metadata = {}
for idx in range(N):
    metadata[idx] = {
        "row_index": int(idx),
        "question": df.at[idx, "question"][:300] if pd.notna(df.at[idx, "question"]) else "",
        "answer": df.at[idx, "answer"][:1000] if pd.notna(df.at[idx, "answer"]) else "",
        "specialty": df.at[idx, "specialty"] if "specialty" in df.columns else None
    }

# 3. Load embedding model
print("Loading embedding model:", EMBEDDING_MODEL_NAME)
embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)

# 4. Compute embeddings in batches
print("Computing embeddings in batches (batch size {})...".format(BATCH_SIZE))
embeddings = np.zeros((N, embedder.get_sentence_embedding_dimension()), dtype='float32')

for start_idx in tqdm(range(0, N, BATCH_SIZE), desc="Embedding batches"):
    end_idx = min(N, start_idx + BATCH_SIZE)
    batch_texts = answers[start_idx:end_idx]
    batch_emb = embedder.encode(batch_texts, show_progress_bar=False, convert_to_numpy=True)
    embeddings[start_idx:end_idx] = batch_emb.astype('float32')

# 5. Normalize embeddings (for cosine similarity using dot product)
#    FAISS IndexFlatIP does inner product; normalized vectors => inner product == cosine similarity
print("Normalizing embeddings...")
faiss.normalize_L2(embeddings)

# 6. Build FAISS index (IndexFlatIP) and wrap with IndexIDMap to preserve doc ids
dim = embeddings.shape[1]
print("Embedding dimension:", dim)
index = faiss.IndexFlatIP(dim)  # inner product index
index_id_map = faiss.IndexIDMap(index)  # allows adding ids

print("Adding vectors to FAISS index...")
index_id_map.add_with_ids(embeddings, np.array(doc_ids, dtype='int64'))

# 7. Save index to disk
print("Saving FAISS index to:", FAISS_INDEX_PATH)
faiss.write_index(index_id_map, FAISS_INDEX_PATH)

# 8. Save metadata (mapping) to JSON for lookup later
print("Saving metadata to:", META_PATH)
with open(META_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("Done. FAISS index and metadata saved.")
print("Index size (n):", index_id_map.ntotal)
print("You can now run retrievals against this index in the next substep.")


c:\Users\USER\Desktop\MediRAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading dataset: neurology_10k_with_rewrites.jsonl
Rows loaded: 10000
Loading embedding model: all-mpnet-base-v2


c:\Users\USER\Desktop\MediRAG\venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\USER\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Computing embeddings in batches (batch size 64)...


Embedding batches: 100%|██████████| 157/157 [44:21<00:00, 16.95s/it]


Normalizing embeddings...
Embedding dimension: 768
Adding vectors to FAISS index...
Saving FAISS index to: answers_faiss.index
Saving metadata to: answers_metadata.json
Done. FAISS index and metadata saved.
Index size (n): 10000
You can now run retrievals against this index in the next substep.


In [2]:
import faiss
import json
import numpy as np
from sentence_transformers import SentenceTransformer

# ----- Paths (same as earlier step) -----
FAISS_INDEX_PATH = "answers_faiss.index"
META_PATH = "answers_metadata.json"
EMBEDDING_MODEL_NAME = "all-mpnet-base-v2"
# ----------------------------------------

# 1. Load FAISS index
print("Loading FAISS index...")
index = faiss.read_index(FAISS_INDEX_PATH)

# 2. Load metadata
print("Loading metadata...")
with open(META_PATH, "r", encoding="utf-8") as f:
    metadata = json.load(f)

# 3. Load embedding model (same as in substep 3.1)
print("Loading embedding model:", EMBEDDING_MODEL_NAME)
embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)

print("FAISS + metadata loaded successfully.")


Loading FAISS index...
Loading metadata...
Loading embedding model: all-mpnet-base-v2
FAISS + metadata loaded successfully.


In [3]:
def retrieve_top_k(query_text, k=3):
    """
    Given a rewritten query string, returns top-k closest documents from FAISS.
    Returns: list of (doc_id, score) pairs.
    """

    # 1. Embed the query
    emb = embedder.encode([query_text], convert_to_numpy=True)
    emb = emb.astype("float32")

    # 2. Normalize (because the FAISS index expects normalized vectors)
    faiss.normalize_L2(emb)

    # 3. Search FAISS
    scores, doc_ids = index.search(emb, k)

    # Convert to Python types
    doc_ids = doc_ids[0].tolist()
    scores = scores[0].tolist()

    # Return list of tuples
    return list(zip(doc_ids, scores))


In [4]:
df = pd.read_json("neurology_10k_with_rewrites.jsonl", lines=True)
print("Loaded rows:", len(df))


Loaded rows: 10000


In [5]:
df["good_rewrite"] = None


In [6]:
CHECKPOINT_FILE = "good_rewrites_partial.jsonl"
SAVE_EVERY = 200   # save every 200 rows to avoid losing work

start_idx = 0

# If a partial file exists, resume from where you left off
if os.path.exists(CHECKPOINT_FILE):
    print("Resuming from checkpoint...")
    df_partial = pd.read_json(CHECKPOINT_FILE, lines=True)
    df["good_rewrite"] = df_partial["good_rewrite"]
    
    # Continue from next row without a good rewrite
    for i in range(len(df)):
        if df.at[i, "good_rewrite"] is None:
            start_idx = i
            break
    print(f"Resuming from row {start_idx}")

else:
    print("Starting fresh evaluation...")


Starting fresh evaluation...


In [7]:
for i in tqdm(range(start_idx, len(df)), desc="Evaluating rewritten queries"):

    rewrites = df.at[i, "rewrites"]

    if not rewrites:
        continue

    best_rewrite = None
    best_score = -1

    # Evaluate ALL rewrites and pick the highest scoring correct one
    for rewrite in rewrites:
        results = retrieve_top_k(rewrite, k=1)
        top_doc_id, score = results[0]

        if top_doc_id == i:
            # This rewrite correctly retrieved the ground-truth doc
            if score > best_score:
                best_score = score
                best_rewrite = rewrite

    # Store the rewrite with highest score (if any matched)
    if best_rewrite is not None:
        df.at[i, "good_rewrite"] = best_rewrite

    # Checkpoint save every N rows
    if i % SAVE_EVERY == 0:
        df_subset = df[["question", "answer", "specialty", "rewrites", "good_rewrite"]]
        df_subset.to_json(CHECKPOINT_FILE, orient="records", lines=True)
        print(f"Checkpoint saved at row {i}")


Evaluating rewritten queries:   0%|          | 2/10000 [00:00<34:12,  4.87it/s]

Checkpoint saved at row 0


Evaluating rewritten queries:   2%|▏         | 202/10000 [00:29<32:17,  5.06it/s]

Checkpoint saved at row 200


Evaluating rewritten queries:   4%|▍         | 402/10000 [01:01<30:55,  5.17it/s]

...

Evaluating rewritten queries:  96%|█████████▌| 9602/10000 [26:22<01:20,  4.96it/s]

Checkpoint saved at row 9600


Evaluating rewritten queries:  98%|█████████▊| 9802/10000 [26:58<00:42,  4.64it/s]

Checkpoint saved at row 9800


Evaluating rewritten queries: 100%|██████████| 10000/10000 [27:33<00:00,  6.05it/s]


In [8]:
OUTPUT_FINAL = "warmup_good_rewrites.jsonl"

df_final = df[ df["good_rewrite"].notna() ]  # keep only rows with good rewrites

df_final.to_json(OUTPUT_FINAL, orient="records", lines=True)
print("🎉 Done! Saved warm-up dataset with good rewrites:", OUTPUT_FINAL)
print("Total good rewrites:", len(df_final))


🎉 Done! Saved warm-up dataset with good rewrites: warmup_good_rewrites.jsonl
Total good rewrites: 7798


In [1]:
import pandas as pd

# Load warm-up dataset from Step 3
INPUT_WARMUP = "warmup_good_rewrites.jsonl"

df = pd.read_json(INPUT_WARMUP, lines=True)

print("Total rows:", len(df))
print("\nColumns:", df.columns.tolist())

# Check missing values
print("\nMissing values per column:")
print(df.isna().sum())

# Preview a few samples
df[["question", "good_rewrite"]].head(5)


Total rows: 7798

Columns: ['question', 'answer', 'specialty', 'rewrites', 'good_rewrite']

Missing values per column:
question        0
answer          0
specialty       0
rewrites        0
good_rewrite    0
dtype: int64


,question,good_rewrite
0,How do BK channels play a role in the excitabi...,BK channels role substantia nigra dopamine neu...
1,How does diffusion imaging help in understandi...,diffusion imaging multiple sclerosis pathophys...
2,What is the minimal clinically important chang...,CID UPDRS total score
3,What are the risk factors associated with the ...,risk factors motor fluctuations Parkinson's
4,What are the common surgical approaches used t...,common surgical approaches carotid-ophthalmic ...


In [2]:
# Substep 4.2 — Build final T5 training pairs

import pandas as pd

INPUT_WARMUP = "warmup_good_rewrites.jsonl"
OUTPUT_TRAIN = "t5_rewriter_train.jsonl"

# Load warm-up dataset
df = pd.read_json(INPUT_WARMUP, lines=True)

print("Loaded rows:", len(df))

# Create T5-style input-output columns
df_train = pd.DataFrame({
    "input_text": "rewrite query: " + df["question"].astype(str),
    "target_text": df["good_rewrite"].astype(str)
})

# Sanity checks
print("\nSample training pair:")
print("INPUT :", df_train.iloc[0]["input_text"])
print("TARGET:", df_train.iloc[0]["target_text"])

# Save final training dataset
df_train.to_json(OUTPUT_TRAIN, orient="records", lines=True)
print("\nSaved T5 training dataset to:", OUTPUT_TRAIN)
print("Final training rows:", len(df_train))


Loaded rows: 7798

Sample training pair:
INPUT : rewrite query: How do BK channels play a role in the excitability of substantia nigra dopamine neurons?

TARGET: BK channels role substantia nigra dopamine neuron excitability

Saved T5 training dataset to: t5_rewriter_train.jsonl
Final training rows: 7798


In [3]:
# Substep 4.3 — Train / Validation split

import pandas as pd
from sklearn.model_selection import train_test_split

INPUT_TRAIN = "t5_rewriter_train.jsonl"

OUTPUT_TRAIN_SPLIT = "t5_rewriter_train_split.jsonl"
OUTPUT_VAL_SPLIT = "t5_rewriter_val_split.jsonl"

# Load dataset
df = pd.read_json(INPUT_TRAIN, lines=True)
print("Total rows:", len(df))

# Shuffle + split
train_df, val_df = train_test_split(
    df,
    test_size=0.10,
    random_state=42,
    shuffle=True
)

print("Training rows:", len(train_df))
print("Validation rows:", len(val_df))

# Save splits
train_df.to_json(OUTPUT_TRAIN_SPLIT, orient="records", lines=True)
val_df.to_json(OUTPUT_VAL_SPLIT, orient="records", lines=True)

print("\nSaved:")
print(" - Training split:", OUTPUT_TRAIN_SPLIT)
print(" - Validation split:", OUTPUT_VAL_SPLIT)

# Quick sanity check
print("\nSample TRAIN pair:")
print(train_df.iloc[0]["input_text"])
print("→", train_df.iloc[0]["target_text"])

print("\nSample VAL pair:")
print(val_df.iloc[0]["input_text"])
print("→", val_df.iloc[0]["target_text"])


Total rows: 7798
Training rows: 7018
Validation rows: 780

Saved:
 - Training split: t5_rewriter_train_split.jsonl
 - Validation split: t5_rewriter_val_split.jsonl

Sample TRAIN pair:
rewrite query: What are some of the surgical approaches that have been suggested for accessing the atrium of the lateral ventricle?

→ accessing atrium lateral ventricle surgery

Sample VAL pair:
rewrite query: What are the potential effects of poor sleep on the development of Alzheimer's disease (AD)?

→ poor sleep impact Alzheimer's development
